In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!pip install -q unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 25.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 103.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 89.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 80.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 79.2 MB/s eta 0:00:00:00:01
 

In [2]:
from unsloth import FastLanguageModel
import torch, json

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)
print("Judge loaded.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


Judge loaded.


In [3]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/ayushsrivastava1805/full-item-for-judge-llm/judge_items_full.json
/kaggle/input/datasets/ayushsrivastava1805/llm-as-judge/judge_items.json


In [4]:
JUDGE_ITEMS = "/kaggle/input/datasets/ayushsrivastava1805/full-item-for-judge-llm/judge_items_full.json"  
items = json.load(open(JUDGE_ITEMS))
print(f"Total items: {len(items)}")

def make_judge_prompt(it):
    return f"""You are evaluating two code review comments for a code change.

CODE CHANGE:
{it['patch']}

REFERENCE (human reviewer's comment):
{it['actual']}

COMMENT 1:
{it['comment_1']}

COMMENT 2:
{it['comment_2']}

Which comment is closer to the reference and more useful as a code review?
Answer with ONLY one word: "1", "2", or "Tie"."""

def judge(it):
    prompt = make_judge_prompt(it)
    msgs = [{"role":"user","content":prompt}]
    inputs = tokenizer.apply_chat_template(msgs, return_tensors="pt", add_generation_prompt=True).to("cuda")
    with torch.no_grad():
        out = model.generate(inputs, max_new_tokens=10, temperature=0.1, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    resp = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()
    if "1" in resp and "2" not in resp: return "1"
    if "2" in resp and "1" not in resp: return "2"
    if "tie" in resp.lower(): return "Tie"
    return resp[:5]

Total items: 1233


In [5]:
from tqdm import tqdm

results = []
for i, it in enumerate(tqdm(items, desc="Judging")):
    verdict = judge(it)
    if verdict in ["1","2"]:
        winner = it["pos_map"][verdict]
    else:
        winner = "Tie"
    results.append({"idx": it["idx"], "verdict": verdict, "winner": winner})
    # save every 200 (safety if it stops)
    if (i+1) % 200 == 0:
        json.dump(results, open("judge_results_full.json","w"))

json.dump(results, open("judge_results_full.json","w"))
print("Done:", len(results))

Judging:   0%|          | 0/1233 [00:00<?, ?it/s]The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=10) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attent

Done: 1233


In [6]:
from collections import Counter
from scipy import stats

c = Counter(r["winner"] for r in results)
total = len(results)
B, A = c["B"], c["A"]
n = B + A

print("="*45)
print("LLM-JUDGE RESULTS (full test set)")
print("="*45)
print(f"Setup B won: {B} ({100*B/total:.1f}%)")
print(f"Setup A won: {A} ({100*A/total:.1f}%)")
print(f"Tie:         {c['Tie']} ({100*c['Tie']/total:.1f}%)")
print()

res = stats.binomtest(B, n, 0.5, alternative='greater')
print(f"Binomial test (B>50%): p={res.pvalue:.4f}")
print(f"Significant? {'YES' if res.pvalue<0.05 else 'NO'}")
ci = res.proportion_ci(confidence_level=0.95)
print(f"95% CI for B win rate: [{ci.low:.3f}, {ci.high:.3f}]")

LLM-JUDGE RESULTS (full test set)
Setup B won: 631 (51.2%)
Setup A won: 576 (46.7%)
Tie:         26 (2.1%)

Binomial test (B>50%): p=0.0600
Significant? NO
95% CI for B win rate: [0.499, 1.000]
